In [1]:
# !pip install scikit-optimize

### Logistic Regression

In [2]:
import pandas as pd
import numpy as np

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

### 1. 사전 작업


In [3]:
# ADASYN 적용된 학습 데이터 로드
train_df = pd.read_csv(r"../../data/processed/ADASYN/data_train_adasyn.csv")

print("shape:", train_df.shape)
print(train_df.head())
print("\ncolumns:")
print(train_df.columns.tolist())

# 실제 타깃 컬럼명
target_col = "Risk_Label"

# 전체 데이터(X)와 타깃(y) 분리
X = train_df
y = train_df[target_col].astype(int)

print("\nX shape:", X.shape)
print("y distribution:")
print(y.value_counts())
print(y.value_counts(normalize=True))

# 불균형 데이터 평가를 위한 G-Mean 함수
def gmean_score(y_true, y_pred):
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()

    recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0.0

    return np.sqrt(recall * specificity)

shape: (3761, 7)
   KOSPI 200_PPO_Hist  GJR_VaR_5_t1  KOSPI 200_OG  KOSPI 200_DMI14  \
0            0.691619      0.632405      0.728036         0.681672   
1            0.667715      0.650805      0.655851         0.660920   
2            0.637613      0.667761      0.325430         0.628385   
3            0.638071      0.683924      0.662242         0.669589   
4            0.649099      0.699923      0.671566         0.689658   

   VKOSPI_Close  NASDAQ_return(%)  Risk_Label  
0      0.638029          0.593612           0  
1      0.654370          0.259590           0  
2      0.660560          0.758918           0  
3      0.626145          0.592042           0  
4      0.586036          0.610832           0  

columns:
['KOSPI 200_PPO_Hist', 'GJR_VaR_5_t1', 'KOSPI 200_OG', 'KOSPI 200_DMI14', 'VKOSPI_Close', 'NASDAQ_return(%)', 'Risk_Label']

X shape: (3761, 7)
y distribution:
Risk_Label
1    1902
0    1859
Name: count, dtype: int64
Risk_Label
1    0.505717
0    0.494283
Name: pr

### 2. Valid 기반 파라미터 최적화

In [4]:
# 검증셋 로드(하이퍼파라미터 선택에 사용)
valid_df = pd.read_csv(r"../../data/processed/ADASYN/data_valid.csv")

# train/valid에서 타깃과 피처 분리
X_train_raw = train_df.drop(columns=[target_col]).copy()
y_train_opt = train_df[target_col].astype(int)

X_valid_raw = valid_df.drop(columns=[target_col]).copy()
y_valid_opt = valid_df[target_col].astype(int)

# 입력은 이미 수치형/스케일링 상태이므로 바로 numpy 배열로 사용
X_train_model = X_train_raw.values
X_valid_model = X_valid_raw.values

# ElasticNet LR 하이퍼파라미터 탐색 범위(C, l1_ratio)
c_grid = np.logspace(-2, 2, 15)  # 0.01 ~ 100 범위에서 15개 (로그 스케일)
l1_ratio_grid = np.arange(0.05, 1.0, 0.05)  # 0.05, 0.10, 0.15, ..., 0.95 (20개)

best_score = -1
best_lr_params = None
search_history = []

print(f"탐색 조합 수: {len(c_grid)} × {len(l1_ratio_grid)} = {len(c_grid) * len(l1_ratio_grid)}")

for c_val in c_grid:
    for l1_val in l1_ratio_grid:
        trial_model = LogisticRegression(
            penalty="elasticnet",
            solver="saga",
            l1_ratio=l1_val,
            C=c_val,
            max_iter=5000,
            random_state=42,
            n_jobs=-1
        )
        trial_model.fit(X_train_model, y_train_opt)
        valid_pred = trial_model.predict(X_valid_model)
        valid_gmean = gmean_score(y_valid_opt, valid_pred)

        search_history.append({
            "C": c_val,
            "l1_ratio": l1_val,
            "valid_gmean": valid_gmean
        })

        # valid G-Mean 최대 조합을 최적 파라미터로 채택
        if valid_gmean > best_score:
            best_score = valid_gmean
            best_lr_params = {"C": c_val, "l1_ratio": l1_val}

print("\n[Valid 기반 최적 파라미터]")
print(f"Best C: {best_lr_params['C']:.4f}, Best l1_ratio: {best_lr_params['l1_ratio']:.2f}")
print(f"Best Valid G-Mean: {best_score:.4f}")

# 상위 5개 조합 확인
search_df = pd.DataFrame(search_history).sort_values("valid_gmean", ascending=False)
print("\n[상위 5개 파라미터]")
print(search_df.head(5))

# valid set 기반 최적 cutoff 탐색 (G-Mean 최대)
cutoff_model = LogisticRegression(
    penalty="elasticnet",
    solver="saga",
    l1_ratio=best_lr_params['l1_ratio'],
    C=best_lr_params['C'],
    max_iter=5000,
    random_state=42,
    n_jobs=-1
)
cutoff_model.fit(X_train_model, y_train_opt)
valid_prob = cutoff_model.predict_proba(X_valid_model)[:, 1]

threshold_grid = np.arange(0.05, 0.951, 0.01)
best_cutoff = 0.5
best_cutoff_gmean = -1

for thr in threshold_grid:
    valid_pred_thr = (valid_prob >= thr).astype(int)
    gm = gmean_score(y_valid_opt, valid_pred_thr)
    if gm > best_cutoff_gmean:
        best_cutoff_gmean = gm
        best_cutoff = float(thr)

print("\n[Valid 기반 최적 Cutoff]")
print(f"Best threshold: {best_cutoff:.2f}")
print(f"Best Valid G-Mean at threshold: {best_cutoff_gmean:.4f}")

# 다음 셀에서 재사용할 변수 보관
X_train_feature_names = X_train_raw.columns
y_train = y_train_opt
y_valid = y_valid_opt

탐색 조합 수: 15 × 19 = 285


c:\Users\cj020\OneDrive\Labtop\KDISS\TS_RL_proj\venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
c:\Users\cj020\OneDrive\Labtop\KDISS\TS_RL_proj\venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1184: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)
c:\Users\cj020\OneDrive\Labtop\KDISS\TS_RL_proj\venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' in


[Valid 기반 최적 파라미터]
Best C: 1.0000, Best l1_ratio: 0.05
Best Valid G-Mean: 0.6733

[상위 5개 파라미터]
            C  l1_ratio  valid_gmean
133  1.000000      0.05     0.673335
134  1.000000      0.10     0.673335
135  1.000000      0.15     0.672868
136  1.000000      0.20     0.672401
131  0.517947      0.90     0.670360

[Valid 기반 최적 Cutoff]
Best threshold: 0.50
Best Valid G-Mean at threshold: 0.6733


c:\Users\cj020\OneDrive\Labtop\KDISS\TS_RL_proj\venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
c:\Users\cj020\OneDrive\Labtop\KDISS\TS_RL_proj\venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1184: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)
c:\Users\cj020\OneDrive\Labtop\KDISS\TS_RL_proj\venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' in

### 3. 최적 파라미터로 로지스틱 회귀 학습

In [5]:
# 이전 셀에서 생성된 필수 변수 점검
required_vars = ['best_lr_params', 'X_train_model', 'y_train', 'X_train_feature_names']
missing_vars = [v for v in required_vars if v not in globals()]

best_C = best_lr_params['C']
best_l1_ratio = best_lr_params['l1_ratio']
print(f"Valid 최적 파라미터 사용 -> C={best_C:.4f}, l1_ratio={best_l1_ratio:.2f}")

# 최적 파라미터로 최종 학습용 입력 정의
X_fit = X_train_model
y_fit = y_train
feature_names = X_train_feature_names

model = LogisticRegression(
    penalty="elasticnet",
    solver="saga",
    l1_ratio=best_l1_ratio,
    C=best_C,
    max_iter=5000,
    random_state=42,
    n_jobs=-1
)

# 최종 로지스틱 회귀 모델 학습
model.fit(X_fit, y_fit)
print("\n로지스틱 회귀 학습 완료")

Valid 최적 파라미터 사용 -> C=1.0000, l1_ratio=0.05

로지스틱 회귀 학습 완료


c:\Users\cj020\OneDrive\Labtop\KDISS\TS_RL_proj\venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
c:\Users\cj020\OneDrive\Labtop\KDISS\TS_RL_proj\venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1184: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)


### 4. 예측 및 성능평가 (train data에서)

In [6]:
# 학습(train) 데이터 기준 확률 계산 후 최적 cutoff 적용
cutoff_to_use = best_cutoff if 'best_cutoff' in globals() else 0.5
y_prob = model.predict_proba(X_fit)[:, 1]
y_pred = (y_prob >= cutoff_to_use).astype(int)

# 핵심 분류 지표 계산
acc = accuracy_score(y_fit, y_pred)
prec = precision_score(y_fit, y_pred, zero_division=0)
rec = recall_score(y_fit, y_pred, zero_division=0)
f1 = f1_score(y_fit, y_pred, zero_division=0)
gmean = gmean_score(y_fit, y_pred)

print("\n[TRAIN performance]")
print(f"Cutoff   : {cutoff_to_use:.2f}")
print(f"Accuracy : {acc:.4f}")
print(f"Precision: {prec:.4f}")
print(f"Recall   : {rec:.4f}")
print(f"F1-score : {f1:.4f}")
print(f"G-Mean   : {gmean:.4f}")

print("\nConfusion Matrix")
print(confusion_matrix(y_fit, y_pred))

print("\nClassification Report")
print(classification_report(y_fit, y_pred, digits=4, zero_division=0))

# valid 데이터 성능도 함께 확인(최적 cutoff 동일 적용)
if 'X_valid_model' in globals() and 'y_valid' in globals():
    y_valid_prob = model.predict_proba(X_valid_model)[:, 1]
    y_valid_pred = (y_valid_prob >= cutoff_to_use).astype(int)
    valid_acc = accuracy_score(y_valid, y_valid_pred)
    valid_gmean = gmean_score(y_valid, y_valid_pred)
    print("\n[VALID performance]")
    print(f"Cutoff   : {cutoff_to_use:.2f}")
    print(f"Accuracy : {valid_acc:.4f}")
    print(f"G-Mean   : {valid_gmean:.4f}")

# 절대계수 기준으로 영향력이 큰 피처 확인
coef_df = pd.DataFrame({
    "feature": feature_names,
    "coef": model.coef_[0],
    "abs_coef": np.abs(model.coef_[0])
}).sort_values("abs_coef", ascending=False)

print("\n상위 계수")
print(coef_df.head(20))


[TRAIN performance]
Cutoff   : 0.50
Accuracy : 0.6676
Precision: 0.6998
Recall   : 0.6004
F1-score : 0.6463
G-Mean   : 0.6650

Confusion Matrix
[[1369  490]
 [ 760 1142]]

Classification Report
              precision    recall  f1-score   support

           0     0.6430    0.7364    0.6866      1859
           1     0.6998    0.6004    0.6463      1902

    accuracy                         0.6676      3761
   macro avg     0.6714    0.6684    0.6664      3761
weighted avg     0.6717    0.6676    0.6662      3761


[VALID performance]
Cutoff   : 0.50
Accuracy : 0.6737
G-Mean   : 0.6733

상위 계수
              feature      coef  abs_coef
5    NASDAQ_return(%) -8.003157  8.003157
4        VKOSPI_Close  2.586729  2.586729
3     KOSPI 200_DMI14 -1.629432  1.629432
2        KOSPI 200_OG -1.411548  1.411548
1        GJR_VaR_5_t1  1.122566  1.122566
0  KOSPI 200_PPO_Hist -0.547806  0.547806


### 5. test data 성능 평가

In [7]:
# test 데이터 로드
test_df = pd.read_csv(r"../../data/processed/ADASYN/data_test.csv")

# 학습 피처 순서와 동일하게 정렬(누락 컬럼은 0으로 채움)
X_test_df = test_df.drop(columns=[target_col]).copy()
X_test_df = X_test_df.reindex(columns=feature_names, fill_value=0)
y_test = test_df[target_col].astype(int)

X_test = X_test_df.values

# test 확률 계산 후 최적 cutoff 적용
cutoff_to_use = best_cutoff if 'best_cutoff' in globals() else 0.5
y_test_prob = model.predict_proba(X_test)[:, 1]
y_test_pred = (y_test_prob >= cutoff_to_use).astype(int)

# test 지표 계산
test_acc = accuracy_score(y_test, y_test_pred)
test_prec = precision_score(y_test, y_test_pred, zero_division=0)
test_rec = recall_score(y_test, y_test_pred, zero_division=0)
test_f1 = f1_score(y_test, y_test_pred, zero_division=0)
test_gmean = gmean_score(y_test, y_test_pred)

print("\n[TEST performance]")
print(f"Cutoff   : {cutoff_to_use:.2f}")
print(f"Accuracy : {test_acc:.4f}")
print(f"Precision: {test_prec:.4f}")
print(f"Recall   : {test_rec:.4f}")
print(f"F1-score : {test_f1:.4f}")
print(f"G-Mean   : {test_gmean:.4f}")

print("\nConfusion Matrix")
print(confusion_matrix(y_test, y_test_pred))

print("\nClassification Report")
print(classification_report(y_test, y_test_pred, digits=4, zero_division=0))


[TEST performance]
Cutoff   : 0.50
Accuracy : 0.6910
Precision: 0.2086
Recall   : 0.6304
F1-score : 0.3135
G-Mean   : 0.6637

Confusion Matrix
[[510 220]
 [ 34  58]]

Classification Report
              precision    recall  f1-score   support

           0     0.9375    0.6986    0.8006       730
           1     0.2086    0.6304    0.3135        92

    accuracy                         0.6910       822
   macro avg     0.5731    0.6645    0.5571       822
weighted avg     0.8559    0.6910    0.7461       822

